In [1]:
# =========================
# SafeSeniors - Small NN Pipeline
# For flattened time-series sensor dataset
# label + 960 features
# =========================

import os
import json
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score
)

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint


In [2]:
# =========================
# 1. CONFIG
# =========================

DATA_PATH = "/Users/dhanujiamanda/Documents/IIT/Stage 3/Edge/CW/SafeSeniors/data/full_dataset.csv"
OUTPUT_DIR = "outputs_nn_final"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.20
VAL_SIZE = 0.15
BATCH_SIZE = 32
EPOCHS = 50

LABEL_COL = "label"

# Set this True if you want only:
# fall -> Fall
# everything else -> Normal
BINARY_MODE = True


In [3]:
# =========================
# 2. LOAD DATA
# =========================

data = pd.read_csv(DATA_PATH)

print("Data shape:", data.shape)
print("\nFirst 5 rows:")
print(data.head())

if LABEL_COL not in data.columns:
    raise ValueError(f"'{LABEL_COL}' column not found in dataset.")

# =========================
# 3. PREPARE LABELS
# =========================

if BINARY_MODE:
    data[LABEL_COL] = data[LABEL_COL].astype(str).str.lower().apply(
        lambda x: "Fall" if x == "fall" else "Normal"
    )
else:
    # Keep original classes
    data[LABEL_COL] = data[LABEL_COL].astype(str)

print("\nLabel distribution:")
print(data[LABEL_COL].value_counts())

# =========================
# 4. FEATURES / LABELS
# =========================

y = data[LABEL_COL]
X = data.drop(columns=[LABEL_COL])

# ensure all features are numeric
X = X.apply(pd.to_numeric, errors="coerce")

# remove rows with invalid feature values
valid_mask = ~X.isna().any(axis=1)
X = X.loc[valid_mask].copy()
y = y.loc[valid_mask].copy()

print("\nFeature matrix shape:", X.shape)
print("Label vector shape:", y.shape)

feature_cols = X.columns.tolist()
joblib.dump(feature_cols, os.path.join(OUTPUT_DIR, "feature_cols.pkl"))


Data shape: (2480, 961)

First 5 rows:
    label  acc_x_0  acc_x_1  acc_x_2  acc_x_3  acc_x_4  acc_x_5  acc_x_6  \
0    idle   0.0000   0.0000   0.0000   0.0000   0.0000   0.0000   0.0000   
1    fall  -0.0008   0.0181   0.1298   0.1815   0.1867   0.0526   0.0188   
2    step   0.0200   0.0200   0.0276   0.0201   0.0000   0.0000   0.0000   
3  motion  -0.0348  -0.0104   0.0309   0.0373   0.0498   0.0600   0.0984   
4    step   0.0200   0.0172   0.0100   0.0020  -0.0099   0.0024   0.0155   

   acc_x_7  acc_x_8  ...  gy_z_150  gy_z_151  gy_z_152  gy_z_153  gy_z_154  \
0   0.0000   0.0000  ...    1.6440    1.6856    1.5975    1.4654    1.5300   
1   0.0228   0.3152  ...    1.2886    1.2177    1.3184    1.1846    0.7587   
2   0.0107   0.0291  ...   -0.5565   -0.1801   -0.2183   -0.3190   -0.3448   
3   0.0810   0.0559  ...   -0.5500   -0.5500   -0.2313   -0.1530   -0.1978   
4   0.0357   0.0200  ...   -0.5644   -0.5155   -0.5806   -0.6057   -0.5801   

   gy_z_155  gy_z_156  gy_z_157  gy

['outputs_nn_final/feature_cols.pkl']

In [4]:
# =========================
# 5. TRAIN / VAL / TEST SPLIT
# =========================

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

val_relative_size = VAL_SIZE / (1.0 - TEST_SIZE)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=val_relative_size,
    random_state=RANDOM_STATE,
    stratify=y_train_full
)

print("\nSplit shapes:")
print("Train:", X_train.shape, y_train.shape)
print("Val:  ", X_val.shape, y_val.shape)
print("Test: ", X_test.shape, y_test.shape)

# =========================
# 6. LABEL ENCODING
# =========================

label_encoder = LabelEncoder()

y_train_enc = label_encoder.fit_transform(y_train)
y_val_enc = label_encoder.transform(y_val)
y_test_enc = label_encoder.transform(y_test)

class_names = label_encoder.classes_
num_classes = len(class_names)

label_mapping = {label: int(idx) for idx, label in enumerate(class_names)}
print("\nLabel mapping:")
print(label_mapping)

joblib.dump(label_encoder, os.path.join(OUTPUT_DIR, "label_encoder.pkl"))
with open(os.path.join(OUTPUT_DIR, "label_mapping.json"), "w") as f:
    json.dump(label_mapping, f, indent=2)

# =========================
# 7. SCALE FEATURES
# =========================

scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_val_scaled = scaler.transform(X_val).astype(np.float32)
X_test_scaled = scaler.transform(X_test).astype(np.float32)

joblib.dump(scaler, os.path.join(OUTPUT_DIR, "scaler.pkl"))

print("\nScaled feature shapes:")
print("Train:", X_train_scaled.shape)
print("Val:  ", X_val_scaled.shape)
print("Test: ", X_test_scaled.shape)

# =========================
# 8. BUILD SMALL NN
# =========================

input_dim = X_train_scaled.shape[1]

model = Sequential([
    Input(shape=(input_dim,)),
    Dense(128, activation="relu"),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dropout(0.2),
])

if num_classes == 2:
    model.add(Dense(1, activation="sigmoid"))
    loss_fn = "binary_crossentropy"
    y_train_nn = y_train_enc
    y_val_nn = y_val_enc
    y_test_nn = y_test_enc
else:
    model.add(Dense(num_classes, activation="softmax"))
    loss_fn = "sparse_categorical_crossentropy"
    y_train_nn = y_train_enc
    y_val_nn = y_val_enc
    y_test_nn = y_test_enc

model.compile(
    optimizer="adam",
    loss=loss_fn,
    metrics=["accuracy"]
)

print("\nModel summary:")
model.summary()


Split shapes:
Train: (1612, 960) (1612,)
Val:   (372, 960) (372,)
Test:  (496, 960) (496,)

Label mapping:
{'Fall': 0, 'Normal': 1}

Scaled feature shapes:
Train: (1612, 960)
Val:   (372, 960)
Test:  (496, 960)

Model summary:


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │       123,008 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 131,329 (513.00 KB)

 Trainable params: 131,329 (513.00 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
# =========================
# 9. TRAIN
# =========================

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),
    ModelCheckpoint(
        filepath=os.path.join(OUTPUT_DIR, "best_model.keras"),
        monitor="val_loss",
        save_best_only=True
    )
]

history = model.fit(
    X_train_scaled,
    y_train_nn,
    validation_data=(X_val_scaled, y_val_nn),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

# =========================
# 10. EVALUATE
# =========================

if num_classes == 2:
    y_prob = model.predict(X_test_scaled).ravel()
    y_pred = (y_prob >= 0.5).astype(int)
else:
    y_prob = model.predict(X_test_scaled)
    y_pred = np.argmax(y_prob, axis=1)

acc = accuracy_score(y_test_enc, y_pred)
f1 = f1_score(y_test_enc, y_pred, average="macro")
precision = precision_score(y_test_enc, y_pred, average="macro")
recall = recall_score(y_test_enc, y_pred, average="macro")

print("\n=== TEST METRICS ===")
print(f"Accuracy : {acc:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_enc, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test_enc,
    y_pred,
    target_names=[str(c) for c in class_names],
    digits=4
))

Epoch 1/50
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.6847 - loss: 0.6174 - val_accuracy: 0.7500 - val_loss: 0.5457
Epoch 2/50
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7608 - loss: 0.5197 - val_accuracy: 0.7581 - val_loss: 0.4461
Epoch 3/50
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7638 - loss: 0.4750 - val_accuracy: 0.8817 - val_loss: 0.3379
Epoch 4/50
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8596 - loss: 0.3482 - val_accuracy: 0.8575 - val_loss: 0.2787
Epoch 5/50
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9376 - loss: 0.2147 - val_accuracy: 0.9597 - val_loss: 0.1216
Epoch 6/50
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9619 - loss: 0.1490 - val_accuracy: 0.9812 - val_loss: 0.0741
Epoch 7/50
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9833 - loss: 0.0884 - val_accuracy: 0.9812 - val_loss: 0.0669
Epoch 8/50
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9796 - loss: 0.0830 - val_accuracy: 0.9812 - val_loss:

In [6]:
# =========================
# 11. SAVE MODEL
# =========================

saved_model_dir = os.path.join(OUTPUT_DIR, "saved_model")
tf.saved_model.save(model, saved_model_dir)

print(f"\nSavedModel exported to: {saved_model_dir}")

# =========================
# 12. CONVERT TO TFLITE
# =========================

converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

tflite_path = os.path.join(OUTPUT_DIR, "fall_detection_model.tflite")
with open(tflite_path, "wb") as f:
    f.write(tflite_model)

print(f"TFLite model saved to: {tflite_path}")

# =========================
# 13. QUICK TFLITE TEST
# =========================

interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

sample = X_test_scaled[:1].astype(np.float32)

interpreter.set_tensor(input_details[0]["index"], sample)
interpreter.invoke()
tflite_output = interpreter.get_tensor(output_details[0]["index"])

print("\nSample TFLite prediction:", tflite_output)

INFO:tensorflow:Assets written to: outputs_nn_final/saved_model/assets


INFO:tensorflow:Assets written to: outputs_nn_final/saved_model/assets



SavedModel exported to: outputs_nn_final/saved_model
TFLite model saved to: outputs_nn_final/fall_detection_model.tflite

Sample TFLite prediction: [[0.5]]


W0000 00:00:1776487582.532235  590090 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1776487582.532251  590090 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
2026-04-18 10:16:22.532587: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: outputs_nn_final/saved_model
2026-04-18 10:16:22.533206: I tensorflow/cc/saved_model/reader.cc:52] Reading meta graph with tags { serve }
2026-04-18 10:16:22.533223: I tensorflow/cc/saved_model/reader.cc:147] Reading SavedModel debug info (if present) from: outputs_nn_final/saved_model
I0000 00:00:1776487582.537036  590090 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
2026-04-18 10:16:22.537897: I tensorflow/cc/saved_model/loader.cc:236] Restoring SavedModel bundle.
2026-04-18 10:16:22.557740: I tensorflow/cc/saved_model/loader.cc:220] Running initialization op on SavedModel bundle at path: outputs_nn_final/saved_model
2026-04-18 10:16:22.564018: I tensorflow/cc/sa

In [7]:
# =========================
# 14. SIMPLE INFERENCE FUNCTION
# =========================

def predict_single_sample(sample_array, scaler_obj, keras_model, encoder):
    """
    sample_array: numpy array of shape (960,)
    """
    sample_array = np.array(sample_array).reshape(1, -1)
    sample_scaled = scaler_obj.transform(sample_array).astype(np.float32)

    output = keras_model.predict(sample_scaled, verbose=0)

    if len(encoder.classes_) == 2:
        prob_fall = float(output.ravel()[0])
        pred = int(prob_fall >= 0.5)
        label = encoder.inverse_transform([pred])[0]
        return {
            "prediction": int(pred),
            "label": str(label),
            "score": prob_fall
        }
    else:
        pred = int(np.argmax(output, axis=1)[0])
        label = encoder.inverse_transform([pred])[0]
        score = float(np.max(output))
        return {
            "prediction": int(pred),
            "label": str(label),
            "score": score
        }

# Example using one actual sample from test set
example_sample = X_test.iloc[0].values
result = predict_single_sample(example_sample, scaler, model, label_encoder)

print("\nExample inference:")
print(result)


Example inference:
{'prediction': 1, 'label': 'Normal', 'score': 0.9945555329322815}


/Users/dhanujiamanda/Documents/IIT/Stage 3/Edge/CW/SafeSeniors/ENV/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(
